# What's in a Word: Breaking Texts and Making Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UW-Madison-Digital-Scholarship-Hub/text-and-data/blob/main/03-Tokenization/Whats-in-a-Word.ipynb)

Before text can be analyzed computationally, it has to be taken apart—split into pieces, normalized, stripped of noise. 

In this notebook, we'll introduces the preprocessing behind nearly all text analysis: tokenization, lemmatization, stemming, and stop word removal. We'll also distinguish this sense of "token" from how the word is used in large language models, a difference that reveals a lot about both. Our goal is to understand not just how to prepare text, but what each choice quietly decides.

> **Important Step**

Run the cells in order from top to bottom. Each cell depends on the ones above it. You don't need a GPU for this notebook; Colab's default CPU runtime is fine.

In [ ]:
""" If you're working in Colab, run this cell to bring everything, including the
saved copies of today's books, over from the GitHub repository. It's safe to re-run."""
import os # lets us check which folder we're in
if not os.path.exists('Whats-in-a-Word.ipynb'): # already in the right folder? then there's nothing to do
    if not os.path.exists('text-and-data'): # only download the repository once
        !git clone https://github.com/UW-Madison-Digital-Scholarship-Hub/text-and-data.git
    %cd text-and-data/03-Tokenization

## Packages
We're going to work with a set of Python packages for helping us access and manipulate textual data, including: 
- [**nltk**](https://www.nltk.org/): The Natural Language Toolkit, one of the most common packages for 
learning natural language processing
- [**spaCy**](https://spacy.io/): A faster, production oriented package for NLP tasks
- [**pandas**](https://pandas.pydata.org/): An open source data manipulation and analysis package
- [**gutenbgerpy**](https://pypi.org/project/gutenbergpy/): A library for filtering, downloading, and querying Project 
Gutenberg data

In [ ]:
# Packages to install if necessary
# nltk, spaCy, and pandas come preinstalled in Colab, but gutenbergpy doesn't
%pip install -q nltk spacy pandas gutenbergpy

# spaCy also needs a trained English model, so we download it if it isn't already installed
import spacy
if not spacy.util.is_package("en_core_web_sm"):
    spacy.cli.download("en_core_web_sm")
import nltk

## What's a Word?

## Tokenization

In [42]:
import nltk
nltk.download('punkt_tab') # import the tokenization dataset from NLTK
from nltk.tokenize import word_tokenize # import the word tokenization module

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/bkrien/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [44]:
open0 = 'It was the best of times, it was the worst of times.'
tokens0 = word_tokenize(open0)
print(tokens0)

['It', 'was', 'the', 'best', 'of', 'times', ',', 'it', 'was', 'the', 'worst', 'of', 'times', '.']


In [45]:
open1 = 'Call me Ishmael.'
tokens1 = word_tokenize(open1)
print(tokens1)

['Call', 'me', 'Ishmael', '.']


## Stemming and Lemmatization

Stemming and Lemmatization are Natural Language Processing techniques for reducing derived for inflected words into a common base form for tasks such as counting. 

**Stemming** uses some basic heuristics to reduce common word forms to their root form or "stem." Note that not all stems are valid word forms (as we'll see). 
**Lemmatization** uses more sophisticated rules to return a valid dictionary form of a root word or "lemma."

For instance, imagine a sentence like "He thought the Lakeshore Path was his favorite run of all the runs he had done this year." 

We have the words "run" and "runs" as the singular and plural forms of the same noun. If we were counting words, we'd likely want to count them as versions of the same word. This is where stemming and lemmatization come in as processes to simplify word forms.

Lets look at an example.

In [49]:
sent0 = 'The running man thought the Lakeshore Path was his favorite run of all the runs he had done this year'
sent_tokens0 = word_tokenize(sent0) #tokenize the sentence into a list of tokens

from nltk.stem import PorterStemmer
porter = PorterStemmer()

for token in sent_tokens0:
    print(porter.stem(token))

the
run
man
thought
the
lakeshor
path
wa
hi
favorit
run
of
all
the
run
he
had
done
thi
year


We can see that this correctly simplified the nouns "run" and "runs" into the common stem of "run" but it has also condensed the adjective "running" into "run," which may not be what we want. Additionally, it cuts off some other words in ways that are usable but somewhat crude.

Stemming is essentially a quick and dirty shortening technique. It requires less processing power and time to run but the results are less nuanced than lemmatization. 

Let's take a look at how lemmatization handles this sentence: 

In [52]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

print(lemmatizer.lemmatize('running'))

[nltk_data] Downloading package wordnet to /Users/bkrien/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


running


In [53]:
for token in sent_tokens0:
    print(lemmatizer.lemmatize(token))

The
running
man
thought
the
Lakeshore
Path
wa
his
favorite
run
of
all
the
run
he
had
done
this
year


## Part-of-Speech Tagging

## Stop Words

## Book-Length Example

Looking at the level of a sentence, it's clear that stop words carry relatively little meaning (in most contexts, at least). But what about at the level of a book? Let's take a look at a few examples. 

We'll use the `gutenbergpy` library, a library for querying and downloading books directly from Project Gutenberg. It allows us to pull the text of books directly from Project Gutenberg directly, using the book's ID number, which can be found at the end of the URL for a specific book (e.g., the URL for one version of *Moby-Dick* is `https://www.gutenberg.org/ebooks/2701` and its ID number is `2701`).

One of the nice things about this particular package si that it allows you to download the text of the book without the Project Gutenberg headings and additional information, which saves a cleaning step. 

In [ ]:
# Let's start by getting a book from Project Gutenberg
import gutenbergpy.textget # Import the module that allows us to download the texts

dickens_id = 98 # ID for the book that we want to select.
dickens_raw = gutenbergpy.textget.get_text_by_id(dickens_id) # book, with header and license
dickens_clean = gutenbergpy.textget.strip_headers(dickens_raw) # removes headers and license

dickens = dickens_clean.decode('utf-8') # matches encoding to ensure clean text
print(dickens[:200]) # prints first 200 characters of the text

We can start with counting up the total number of words in the text and looking at the number of words that appear most frequently. We'll use the `pandas` library and simply tally the number of times that individual words appear. 

In [12]:
# We'll create a function that will count the words in a given text
import re
from collections import Counter
import pandas as pd

def word_counts(text):
    """Return a DataFrame of unique words and their counts, most frequent first."""
    # gutenbergpy returns bytes, so decode if needed
    if isinstance(text, bytes):
        text = text.decode("utf-8")

    # lowercase, then pull out words (keeping contractions like "don't")
    words = re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower())

    counts = Counter(words)
    df = pd.DataFrame(counts.most_common(), columns=["word", "count"])
    return df

In [ ]:
# Let's take a look at the top 20 most commons words in A Tale of Two Cities
word_counts(dickens).head(20) # show the top 20 words in the list of words

,word,count
0,the,8053
1,and,4999
2,of,4014
3,to,3571
4,a,2953
5,in,2603
6,it,2067
7,his,2011
8,i,1990
9,that,1941


In [17]:
# We'll create a function that removes stop words from a given text
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True) # downloads NLTK's stop word lists if needed

def remove_stop_words(text, stop_words=None):
    """Return the text as a lowercase string of words with stop words removed."""
    # gutenbergpy returns bytes, so decode if needed
    if isinstance(text, bytes):
        text = text.decode("utf-8")

    # use NLTK's English stop word list unless we pass in our own
    if stop_words is None:
        stop_words = set(stopwords.words('english'))

    # swap curly apostrophes for straight ones so contractions match the stop word list
    text = text.lower().replace("\u2019", "'")

    # split into words the same way word_counts does, then drop the stop words
    words = re.findall(r"[a-z]+(?:'[a-z]+)?", text)
    kept = [word for word in words if word not in stop_words]

    return " ".join(kept) # join back into a string so it can go into word_counts

In [18]:
# Now let's look at the top 20 words in A Tale of Two Cities without the stop words
dickens_no_stops = remove_stop_words(dickens)
word_counts(dickens_no_stops).head(20)

,word,count
0,said,661
1,mr,622
2,one,438
3,would,343
4,lorry,339
5,upon,289
6,man,284
7,could,282
8,defarge,282
9,little,267


### Tagging the Book with spaCy
Counting words tells us *what* words appear, but not *how* they're being used. Part-of-speech (POS) tagging labels each word with its grammatical role (noun, verb, pronoun, etc.), and named entity recognition (NER) picks out names of people, places, organizations, and so on.

We'll use `spaCy` for this. Note that we run spaCy on the **original** text, not the stop-word-free version: the tagger relies on the surrounding words (including stop words) and capitalization to decide what each word is doing.

In [22]:
# We'll create a function that runs spaCy over a whole book
import spacy

# load spaCy's small English model (we don't need the dependency parser, so we turn it off to save time)
nlp = spacy.load("en_core_web_sm", disable=["parser"])

def tag_text(text):
    """Run spaCy over the text one paragraph at a time and return a list of processed Docs."""
    # gutenbergpy returns bytes, so decode if needed
    if isinstance(text, bytes):
        text = text.decode("utf-8")

    # split on blank lines into paragraphs, and join the line breaks inside each paragraph
    # (spaCy can't take a whole book at once, and paragraphs are a natural unit anyway)
    paragraphs = [" ".join(p.split()) for p in re.split(r"\n\s*\n", text) if p.strip()]

    return list(nlp.pipe(paragraphs, batch_size=50))

In [23]:
# Tag A Tale of Two Cities (this takes a minute or so, since spaCy is reading the whole book)
dickens_docs = tag_text(dickens)

In [24]:
# We'll create a function that puts every word and its tags into a DataFrame
def pos_table(docs):
    """Return a DataFrame with one row per word: the word, its part of speech, its detailed tag, and its lemma."""
    rows = []
    for doc in docs:
        for token in doc:
            if token.is_alpha: # skip punctuation, numbers, and whitespace
                rows.append({
                    "word": token.text,
                    "pos": token.pos_, # broad part of speech (NOUN, VERB, PRON, ...)
                    "tag": token.tag_, # more detailed tag (e.g., VBD = past tense verb)
                    "lemma": token.lemma_,
                })
    return pd.DataFrame(rows)

dickens_pos = pos_table(dickens_docs)
dickens_pos.head(20)

,word,pos,tag,lemma
0,A,DET,DT,a
1,TALE,NOUN,NN,tale
2,OF,ADP,IN,of
3,TWO,NUM,CD,two
4,CITIES,NOUN,NNS,city
5,A,DET,DT,a
6,STORY,NOUN,NN,story
7,OF,ADP,IN,of
8,THE,DET,DT,the
9,FRENCH,PROPN,NNP,FRENCH


### Counting Parts of Speech and Places
Now that every word is tagged, we can count just the words in a particular category. spaCy uses the [Universal POS tags](https://universaldependencies.org/u/pos/): `PRON` for pronouns, `VERB` for verbs, `NOUN` for common nouns, `PROPN` for proper nouns, and so on. Note that spaCy tags helping verbs like *was* and *had* (as in "had gone") as `AUX` rather than `VERB`.

For places, we'll use the entity labels `GPE` (countries, cities, states), `LOC` (other locations, like rivers or mountains), and `FAC` (buildings, bridges, streets).

In [25]:
# We'll create a function that counts the words with a given part of speech
def pos_counts(docs, pos):
    """Return a DataFrame of words with the given part(s) of speech and their counts, most frequent first."""
    # accept either a single tag ("NOUN") or a list of tags (["NOUN", "PROPN"])
    if isinstance(pos, str):
        pos = [pos]

    words = [token.text.lower() for doc in docs for token in doc
             if token.pos_ in pos and token.is_alpha]

    counts = Counter(words)
    return pd.DataFrame(counts.most_common(), columns=["word", "count"])

In [26]:
# Pronouns in A Tale of Two Cities
pos_counts(dickens_docs, "PRON").head(20)

,word,count
0,it,2060
1,his,2009
2,i,1957
3,he,1855
4,you,1420
5,her,1043
6,him,972
7,my,655
8,they,567
9,me,528


In [27]:
# Verbs in A Tale of Two Cities
pos_counts(dickens_docs, "VERB").head(20)

,word,count
0,said,658
1,know,231
2,see,197
3,looked,194
4,made,186
5,say,165
6,come,157
7,came,141
8,go,139
9,looking,126


In [28]:
# Nouns in A Tale of Two Cities
pos_counts(dickens_docs, "NOUN").head(20)

,word,count
0,man,297
1,time,262
2,hand,246
3,night,216
4,day,204
5,way,184
6,father,183
7,face,181
8,head,176
9,eyes,163


In [29]:
# We'll create a function that counts the named entities with given labels (by default, places)
def entity_counts(docs, labels=("GPE", "LOC", "FAC")):
    """Return a DataFrame of named entities with the given labels and their counts, most frequent first."""
    entities = [ent.text for doc in docs for ent in doc.ents if ent.label_ in labels]

    counts = Counter(entities)
    return pd.DataFrame(counts.most_common(), columns=["entity", "count"])

# Places in A Tale of Two Cities
entity_counts(dickens_docs).head(20)

,entity,count
0,Paris,62
1,France,58
2,England,38
3,Sydney,32
4,Monseigneur,32
5,London,25
6,Bastille,21
7,Stryver,20
8,Soho,16
9,Dover,15


Take a look through these tables: does the tagger get everything right? Models like spaCy's are trained mostly on modern text (news, web pages), so 19th-century prose can trip them up. Watch for words tagged with the wrong part of speech, and for "places" that are actually people (or vice versa).

### Lemmatization
Lemmatization reduces each word to its dictionary form (its *lemma*): *was* and *were* become *be*, *hands* becomes *hand*, *said* becomes *say*. spaCy already worked out each word's lemma when it tagged the book, so we just need to collect them. We'll join the lemmas back into a single string so that it can be fed into `remove_stop_words` and `word_counts`, just like the original text.

In [30]:
# We'll create a function that turns tagged text into a string of lemmas
def lemmatize(docs):
    """Return the text as a lowercase string of lemmas, ready for remove_stop_words or word_counts."""
    lemmas = [token.lemma_.lower() for doc in docs for token in doc if token.is_alpha]
    return " ".join(lemmas)

dickens_lemmas = lemmatize(dickens_docs)
print(dickens_lemmas[:200])

a tale of two city a story of the french revolution by charles dickens content book the first recall to life chapter i the period chapter ii the mail chapter iii the night shadows chapter iv the prepa


In [31]:
# Compare these to the top 20 words (without stop words) from before
word_counts(remove_stop_words(dickens_lemmas)).head(20)

,word,count
0,say,869
1,look,479
2,one,442
3,go,416
4,see,413
5,know,407
6,man,392
7,hand,374
8,lorry,369
9,come,361


### Putting It All Together
Now we can combine everything into a single workflow: download a book, tag it once with spaCy, and build all of our tables from that one pass. The function returns a dictionary of DataFrames, so we can look at whichever table we want by name.

In [32]:
# We'll create a function that runs the whole workflow on a Project Gutenberg book
def analyze_book(book_id):
    """Download a Project Gutenberg book and return a dictionary of word frequency tables."""
    # 1. download the book and strip the Project Gutenberg header and license
    raw = gutenbergpy.textget.get_text_by_id(book_id)
    text = gutenbergpy.textget.strip_headers(raw).decode("utf-8")

    # 2. tag the book with spaCy (the slow step, so we only do it once)
    docs = tag_text(text)

    # 3. build each table from the original text or the tagged docs
    return {
        "words": word_counts(text),
        "words_no_stops": word_counts(remove_stop_words(text)),
        "lemmas_no_stops": word_counts(remove_stop_words(lemmatize(docs))),
        "pronouns": pos_counts(docs, "PRON"),
        "verbs": pos_counts(docs, "VERB"),
        "nouns": pos_counts(docs, "NOUN"),
        "places": entity_counts(docs),
    }

In [33]:
# Let's try it out on Moby-Dick
moby = analyze_book(2701)

# show the top 10 of each table side by side
pd.concat({name: table.head(10) for name, table in moby.items()}, axis=1)

words        words_no_stops       lemmas_no_stops       pronouns        \
   word  count           word count            word count     word count   
0   the  14535          whale  1110           whale  1543      his  2529   
1    of   6624            one   912             one   941       it  2489   
2   and   6447           like   647             man   774        i  2066   
3     a   4747           upon   566            like   650       he  1880   
4    to   4627            man   479            ship   619      him  1060   
5    in   4184            sea   454             say   610      you   891   
6  that   3085            old   450             see   578     they   666   
7   his   2532           ahab   440            upon   566       me   622   
8    it   2522           ship   439             sea   534    their   620   
9     i   2127             ye   431            ahab   504       my   587   

    verbs         nouns                 places        
     word count    word count           entity count  
0    said   296   whale   879           Pequod   132  
1  seemed   283    ship   508            Flask    73  
2     see   256     man   506        Nantucket    62  
3     say   229     sea   429           Bildad    56  
4      go   191    boat   332             thou    55  
5    made   177    head   330        Leviathan    42  
6    come   173    time   327  the White Whale    32  
7    seen   164     way   267            Jonah    31  
8     let   158  whales   243          Pacific    29  
9    look   157     men   239        Greenland    27

## Your Turn

In [ ]:
# Now you can grab a book of your choice
import gutenbergpy.textget # Import the module that allows us to download the texts

book_id = [YOUR_BOOKS_ID] # ID for the book that we want to select.
book_raw = gutenbergpy.textget.get_text_by_id(book_id) # book, with header and license
book_clean = gutenbergpy.textget.strip_headers(book_raw) # removes headers and license

book = book_clean.decode('utf-8') # matches encoding to ensure clean text
print(book[:200]) # prints first 200 characters of the text